# Kendi ChatGPT'ni eğit
Hazır bir dil modelini **kendi txt dosyanla** ince ayarlayıp (LoRA) telefonda çalışacak hale getirir.

1. Çalışma zamanı > Çalışma zamanı türünü değiştir > **T4 GPU** seç
2. Hücreleri sırayla çalıştır
3. En sonda modelin Hugging Face'e yüklenir, adını web sayfana yazarsın

**txt formatı:** her soru `Kullanıcı: ...`, her cevap `AI: ...` satırında olmalı.

In [ ]:
!pip -q install transformers peft datasets accelerate

## Ayarlar (büyütmek için bunları değiştir)

In [ ]:
MODEL   = "Qwen/Qwen2.5-1.5B-Instruct"  # web sayfasındaki 1.5B modelle uyumlu kalsın
EPOCHS  = 3        # veriyi kaç kez okusun (az veride 5-10 yap)
RANK    = 16       # LoRA boyutu = eğitilen parametre sayısı (8, 16, 32, 64)
LR      = 2e-4
MAXLEN  = 512      # bir konuşmanın en fazla token uzunluğu
HF_REPO = "KULLANICI_ADIN/benim-modelim"  # Hugging Face'te açılacak depo

## 1. txt dosyanı yükle

In [ ]:
from google.colab import files
up = files.upload()
text = "\n".join(open(n, encoding="utf-8").read() for n in up)
pairs, u = [], None
for line in text.splitlines():
    line = line.strip()
    if line.startswith("Kullanıcı:"): u = line[len("Kullanıcı:"):].strip()
    elif line.startswith("AI:") and u:
        pairs.append((u, line[3:].strip())); u = None
print(len(pairs), "soru-cevap bulundu")
assert len(pairs) >= 20, "En az 20, tercihen yüzlerce soru-cevap gerekli"

## 2. Modeli yükle ve LoRA ekle

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, device_map="cuda")
model.gradient_checkpointing_enable(); model.enable_input_require_grads()
cfg = LoraConfig(r=RANK, lora_alpha=2*RANK, lora_dropout=0.05, task_type="CAUSAL_LM",
                 target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])
model = get_peft_model(model, cfg)
for p in model.parameters():
    if p.requires_grad: p.data = p.data.float()
model.print_trainable_parameters()

## 3. Eğit

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
SYS = "Sen Türkçe konuşan, yardımsever bir asistansın."
class DS(torch.utils.data.Dataset):
    def __init__(s):
        s.items = []
        for q, a in pairs:
            t = tok.apply_chat_template([{"role":"system","content":SYS},{"role":"user","content":q},{"role":"assistant","content":a}], tokenize=False)
            s.items.append(tok(t, truncation=True, max_length=MAXLEN))
    def __len__(s): return len(s.items)
    def __getitem__(s, i): return s.items[i]
args = TrainingArguments(output_dir="out", per_device_train_batch_size=4, gradient_accumulation_steps=2,
    num_train_epochs=EPOCHS, learning_rate=LR, fp16=True, logging_steps=10, save_strategy="no", report_to="none",
    warmup_ratio=0.05, lr_scheduler_type="cosine")
Trainer(model=model, args=args, train_dataset=DS(), data_collator=DataCollatorForLanguageModeling(tok, mlm=False)).train()

## 4. Dene

In [ ]:
def sor(q):
    ids = tok.apply_chat_template([{"role":"system","content":SYS},{"role":"user","content":q}], add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = model.generate(ids, max_new_tokens=150, do_sample=True, temperature=0.7)
    return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True)
print(sor("Merhaba, kendini tanıtır mısın?"))

## 5. Birleştir ve telefon formatına çevir (MLC)
Not: MLC paketinin sürümü zamanla değişir. Kurulum hata verirse https://llm.mlc.ai/docs adresindeki güncel komuta bak.

In [ ]:
merged = model.merge_and_unload()
merged.save_pretrained("merged"); tok.save_pretrained("merged")
!pip -q install --pre -U -f https://mlc.ai/wheels mlc-llm-nightly-cu122 mlc-ai-nightly-cu122
!mlc_llm convert_weight ./merged --quantization q4f16_1 -o ./mlc
!mlc_llm gen_config ./merged --quantization q4f16_1 --conv-template qwen2 -o ./mlc

## 6. Hugging Face'e yükle
[huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) adresinden **write** yetkili bir token al.

In [ ]:
from huggingface_hub import login, HfApi
login()
api = HfApi(); api.create_repo(HF_REPO, exist_ok=True)
api.upload_folder(folder_path="mlc", repo_id=HF_REPO)
print("Bitti! Web sayfasındaki 'Kendi modelin' kutusuna şunu yaz:", HF_REPO)